In [17]:
import init_creds as creds
import sys
from langchain_openai import AzureChatOpenAI

sys.path.insert(1, '../../')

# # Configure Azure OpenAI And OpenAI Embeddings
# AZURE_OPENAI_KEY = os.getenv('OPENAI_API_KEY')
# AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT')
# AZURE_OPENAI_API_VERSION = os.getenv('OPENAI_API_VERSION')


AZURE_OPENAI_KEY = creds.get_api_key()
AZURE_OPENAI_ENDPOINT = creds.get_endpoint()
AZURE_OPENAI_API_VERSION = "2025-04-01-preview"

if not AZURE_OPENAI_KEY:
    raise ValueError("No AZURE_OPENAI_KEY set for Azure OpenAI API")
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("No AZURE_OPENAI_ENDPOINT set for Azure OpenAI API")

llm = AzureChatOpenAI(
    model="gpt-4o-mini",
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)


In [ ]:
# Schema for structured output
from pydantic import BaseModel, Field

class SearchQuery(BaseModel):
    search_query: str = Field(None, description="Query that is optimized web search.")
    justification: str = Field(
        None, description="Why this query is relevant to the user's request."
    )

In [18]:
#Augment the LLM with schema for structured output
structured_llm = llm.with_structured_output(SearchQuery)

# Invoke the augmented LLM
output = structured_llm.invoke("How does Calcium CT score relate to high cholesterol?")
print(output.search_query)
print(output.justification)
print(output)

Calcium CT score cholesterol relationship
This query is relevant as it focuses on understanding the connection between calcium coronary artery scoring and cholesterol levels, which is important for assessing cardiovascular risk.
search_query='Calcium CT score cholesterol relationship' justification='This query is relevant as it focuses on understanding the connection between calcium coronary artery scoring and cholesterol levels, which is important for assessing cardiovascular risk.'


In [35]:
# Define a tool
def multiply(a: int, b: int) -> int:
    return a * b

# Augment the LLM with tools
llm_with_tools = llm.bind_tools([multiply])

# Invoke the LLM with input that triggers the tool call
msg = llm_with_tools.invoke("What is 2 times 4?")

# Get the tool call
print(msg)

#print msg content
print(msg.content)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 48, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-CrT04Fxi8RXAuytfd2uqbfgNCSxk2', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}], 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}} id='lc_run--019b60ff-3415-77e0-b50c-496c858a9a50-0' tool_calls=[{'name': 'mu

In [32]:
tool_call = msg.tool_calls[0]
result = multiply(**tool_call['args'])
print(result)

8


In [34]:
 # 3. Pass the result back to the LLM to get the final answer
        # We need to pass the conversation history: [UserMessage, AIMessage (with tool call), ToolMessage (with result)]
from langchain_core.messages import HumanMessage, ToolMessage

messages = [
    HumanMessage(content="What is 2 times 4?"),
    msg,
    ToolMessage(tool_call_id=tool_call['id'], content=str(result))
]

final_response = llm_with_tools.invoke(messages)
print(final_response.content)

2 times 4 is 8.
